In [ ]:
%pip install -q \
    "transformers==4.40.0" \
    "datasets==2.19.0" \
    "accelerate==0.29.3" \
    "librosa==0.10.1" \
    "soundfile==0.12.1" \
    "scikit-learn==1.4.2" \
    "tqdm"
 
print("✓ Dependencies installed")

In [ ]:
# %% — Cell 2: Imports
import os, gc, json, random, time, warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
 
import librosa
from transformers import (
    Wav2Vec2Model,
    AutoFeatureExtractor,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
 
print("✓ All imports successful")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# %% — Cell 3: Configuration  ← EDIT THE PATHS BELOW
CFG = dict(
    # ── Dataset paths — paste the Kaggle input paths after connecting datasets ──
    # RAVDESS (speech only): https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
    #   Typical Kaggle path: /kaggle/input/ravdess-emotional-speech-audio/audio_speech_actors_01-24/
    RAVDESS_ROOT = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio",
 
    # TESS: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess
    #   Typical Kaggle path: /kaggle/input/toronto-emotional-speech-set-tess/data/
    TESS_ROOT    = "/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess",
 
    OUTPUT_DIR   = "/kaggle/working/",
 
    # ── Model ──────────────────────────────────────────────────────────────
    MODEL_ID     = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim",
    NUM_LABELS   = 7,
    HIDDEN_DIM   = 256,
    DROPOUT      = 0.1,
 
    # ── Audio ───────────────────────────────────────────────────────────────
    SAMPLE_RATE  = 16_000,
    MAX_DURATION = 6.0,
 
    # ── Training ────────────────────────────────────────────────────────────
    SEED           = 42,
    TEST_SIZE      = 0.15,
    VAL_SIZE       = 0.15,
    BATCH_SIZE     = 8,       # reduce to 4 if OOM on T4
    GRAD_ACCUM     = 4,       # effective batch = BATCH_SIZE * GRAD_ACCUM = 32
    EPOCHS         = 10,
    LR_HEAD        = 1e-4,
    LR_BASE        = 5e-6,
    WARMUP_RATIO   = 0.10,
    WEIGHT_DECAY   = 0.01,
    UNFREEZE_EPOCH = 3,
    MIXED_PREC     = True,
 
    # ── 7-class label map ────────────────────────────────────────────────────
    # RAVDESS "calm" (02) is merged → Neutral so both datasets share the same 7 classes
    LABEL_MAP = {
        0: "Neutral", 1: "Happy",   2: "Sad",      3: "Angry",
        4: "Fear",    5: "Disgust", 6: "Surprise",
    },
)
 
os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
with open(os.path.join(CFG["OUTPUT_DIR"], "config.json"), "w") as f:
    json.dump(CFG, f, indent=2)
 
random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])
 
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_SAMPLES = int(CFG["MAX_DURATION"] * CFG["SAMPLE_RATE"])
 
print(f"✓ Config saved  |  Device: {DEVICE}")

In [ ]:
# %% — Cell 4: Build dataframe from RAVDESS + TESS filenames (no CSV needed)
# ── RAVDESS filename format: 03-01-03-01-01-01-01.wav
#    segment index [2] (1-based emotion code):
#    01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised
#
# ── TESS folder format: OAF_<emotion>/ or YAF_<emotion>/
#    e.g. OAF_angry, YAF_happy, OAF_neutral, YAF_ps (= surprise)
 
# Map raw emotion strings → unified 7-class integer label
# RAVDESS "calm" (02) is treated as Neutral (merged) — keeps 7 balanced classes
RAVDESS_EMOTION_MAP = {
    "01": 0,   # neutral
    "02": 0,   # calm  → neutral
    "03": 1,   # happy
    "04": 2,   # sad
    "05": 3,   # angry
    "06": 4,   # fearful
    "07": 5,   # disgust
    "08": 6,   # surprised
}
 
TESS_EMOTION_MAP = {
    "neutral": 0,
    "happy":   1,
    "sad":     2,
    "angry":   3,
    "fear":    4,
    "disgust": 5,
    "ps":      6,   # TESS uses "ps" for pleasant-surprise
    "surprise":6,
}
 
records = []
 
# ── Parse RAVDESS ────────────────────────────────────────────────────────────
print("Scanning RAVDESS …")
for actor_dir in sorted(os.listdir(CFG["RAVDESS_ROOT"])):
    actor_path = os.path.join(CFG["RAVDESS_ROOT"], actor_dir)
    if not os.path.isdir(actor_path):
        continue
    for fname in os.listdir(actor_path):
        if not fname.endswith(".wav"):
            continue
        parts = fname.replace(".wav", "").split("-")
        if len(parts) < 7:
            continue
        emotion_code = parts[2]          # e.g. "03"
        if emotion_code not in RAVDESS_EMOTION_MAP:
            continue
        records.append({
            "file_path": os.path.join(actor_path, fname),
            "label":     RAVDESS_EMOTION_MAP[emotion_code],
            "source":    "RAVDESS",
        })
 
print(f"  RAVDESS: {len(records)} files")
 
# ── Parse TESS ───────────────────────────────────────────────────────────────
print("Scanning TESS …")
tess_count = 0
for folder in sorted(os.listdir(CFG["TESS_ROOT"])):
    folder_path = os.path.join(CFG["TESS_ROOT"], folder)
    if not os.path.isdir(folder_path):
        continue
    # folder name like "OAF_angry" or "YAF_ps"
    parts = folder.lower().split("_", 1)
    if len(parts) < 2:
        continue
    emotion_str = parts[1]
    if emotion_str not in TESS_EMOTION_MAP:
        print(f"  ⚠ Unrecognised TESS folder: {folder} — skipping")
        continue
    for fname in os.listdir(folder_path):
        if not fname.endswith(".wav"):
            continue
        records.append({
            "file_path": os.path.join(folder_path, fname),
            "label":     TESS_EMOTION_MAP[emotion_str],
            "source":    "TESS",
        })
        tess_count += 1
 
print(f"  TESS   : {tess_count} files")
 
df = pd.DataFrame(records)
print(f"\n✓ Combined dataset: {len(df):,} samples")
print("\nLabel distribution:")
label_names = CFG["LABEL_MAP"]
dist = df["label"].value_counts().sort_index()
for idx, count in dist.items():
    print(f"  {idx} {label_names[idx]:<10} {count:>5}  ({count/len(df)*100:.1f}%)")
 
print("\nSource breakdown:")
print(df["source"].value_counts().to_string())

In [ ]:
# %% — Cell 4: Dataset class
class SERDataset(Dataset):
    """Loads raw waveforms on-the-fly; no pre-caching to save disk space."""
 
    def __init__(self, df, feature_extractor, augment=False):
        self.df = df.reset_index(drop=True)
        self.fe = feature_extractor
        self.augment = augment
 
    def __len__(self):
        return len(self.df)
 
    def _load_audio(self, path):
        wav, sr = librosa.load(path, sr=CFG["SAMPLE_RATE"], mono=True)
        # Truncate / pad to MAX_SAMPLES
        if len(wav) > MAX_SAMPLES:
            # random crop during training, centre crop at test
            if self.augment:
                start = random.randint(0, len(wav) - MAX_SAMPLES)
                wav = wav[start: start + MAX_SAMPLES]
            else:
                mid = len(wav) // 2
                half = MAX_SAMPLES // 2
                wav = wav[max(0, mid - half): mid - half + MAX_SAMPLES]
        else:
            wav = np.pad(wav, (0, MAX_SAMPLES - len(wav)))
        return wav.astype(np.float32)
 
    def _augment(self, wav):
        # Time-stretch (±10 %)
        if random.random() < 0.3:
            rate = random.uniform(0.9, 1.1)
            wav = librosa.effects.time_stretch(wav, rate=rate)
            wav = wav[:MAX_SAMPLES]
            if len(wav) < MAX_SAMPLES:
                wav = np.pad(wav, (0, MAX_SAMPLES - len(wav)))
        # Additive noise
        if random.random() < 0.4:
            noise = np.random.randn(len(wav)).astype(np.float32) * 0.005
            wav = wav + noise
        return wav
 
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = row["file_path"]   # already absolute (set by Cell 4 parser)
        wav  = self._load_audio(path)
        if self.augment:
            wav = self._augment(wav)
 
        inputs = self.fe(
            wav,
            sampling_rate=CFG["SAMPLE_RATE"],
            return_tensors="pt",
            padding=True,
        )
        return {
            "input_values":  inputs.input_values.squeeze(0),
            "attention_mask": inputs.attention_mask.squeeze(0),
            "labels":        torch.tensor(int(row["label"]), dtype=torch.long),
        }
 
print("✓ Dataset class defined")

In [ ]:
# %% — Cell 5: Custom classification head
class Wav2Vec2RobustClassifier(nn.Module):
    """
    Wraps Wav2Vec2Model (backbone only — no regression head) and adds:
      • Adaptive average-pool over time
      • 2-layer MLP classifier
    """
    def __init__(self, base_model, num_labels=7):
        super().__init__()
        self.wav2vec2   = base_model
        self.pool       = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Linear(1024, CFG["HIDDEN_DIM"]),
            nn.GELU(),
            nn.Dropout(CFG["DROPOUT"]),
            nn.Linear(CFG["HIDDEN_DIM"], num_labels),
        )
        # Class-frequency weighting can be set externally
        self.criterion = nn.CrossEntropyLoss()
 
    def set_criterion(self, class_weights):
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
 
    def freeze_backbone(self):
        for p in self.wav2vec2.parameters():
            p.requires_grad = False
 
    def unfreeze_backbone(self):
        for p in self.wav2vec2.parameters():
            p.requires_grad = True
 
    def forward(self, input_values, attention_mask=None, labels=None):
        out    = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state.transpose(1, 2)   # (B, 1024, T)
        pooled = self.pool(hidden).squeeze(-1)            # (B, 1024)
        logits = self.classifier(pooled)                  # (B, num_labels)
 
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)
        return {"loss": loss, "logits": logits}
 
print("✓ Classification head defined")

In [ ]:
# %% — Cell 6: Load pre-trained backbone + feature extractor
print(f"Loading feature extractor & backbone from: {CFG['MODEL_ID']}")
feature_extractor = AutoFeatureExtractor.from_pretrained(CFG["MODEL_ID"])
base_model        = Wav2Vec2Model.from_pretrained(CFG["MODEL_ID"])
 
model = Wav2Vec2RobustClassifier(base_model, num_labels=CFG["NUM_LABELS"])
model.freeze_backbone()          # stage-1: train head only
model = model.to(DEVICE)
 
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model on {DEVICE}")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}  (backbone frozen)")

In [ ]:
# %% — Cell 7: Split & build data loaders  (df already built in Cell 4)
print(f"Dataset size : {len(df):,} samples")
 
# Stratified split
train_df, temp_df = train_test_split(
    df, test_size=CFG["TEST_SIZE"] + CFG["VAL_SIZE"],
    stratify=df["label"], random_state=CFG["SEED"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=CFG["TEST_SIZE"] / (CFG["TEST_SIZE"] + CFG["VAL_SIZE"]),
    stratify=temp_df["label"], random_state=CFG["SEED"]
)
print(f"Split  → train {len(train_df):,}  |  val {len(val_df):,}  |  test {len(test_df):,}")
 
# Class weights for imbalanced data
label_counts = train_df["label"].value_counts().sort_index().values
class_weights = torch.tensor(
    1.0 / (label_counts / label_counts.sum()), dtype=torch.float32
).to(DEVICE)
model.set_criterion(class_weights)
print("✓ Class weights applied:", class_weights.cpu().numpy().round(3))
 
# DataLoaders
def make_loader(split_df, augment, shuffle):
    ds = SERDataset(split_df, feature_extractor, augment=augment)
    return DataLoader(
        ds, batch_size=CFG["BATCH_SIZE"], shuffle=shuffle,
        num_workers=2, pin_memory=True, drop_last=False,
    )
 
train_loader = make_loader(train_df, augment=True,  shuffle=True)
val_loader   = make_loader(val_df,   augment=False, shuffle=False)
test_loader  = make_loader(test_df,  augment=False, shuffle=False)
print("✓ DataLoaders ready")

In [ ]:
# %% — Cell 8: Optimiser, scheduler, scaler
def build_optimiser(unfreeze=False):
    param_groups = [{"params": model.classifier.parameters(), "lr": CFG["LR_HEAD"]}]
    if unfreeze:
        param_groups.append({"params": model.wav2vec2.parameters(), "lr": CFG["LR_BASE"]})
    return torch.optim.AdamW(param_groups, weight_decay=CFG["WEIGHT_DECAY"])
 
steps_per_epoch = len(train_loader) // CFG["GRAD_ACCUM"]
total_steps     = steps_per_epoch * CFG["EPOCHS"]
warmup_steps    = int(total_steps * CFG["WARMUP_RATIO"])
 
optimiser = build_optimiser(unfreeze=False)
scheduler = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)
scaler    = GradScaler(enabled=CFG["MIXED_PREC"])
 
print(f"✓ Optimiser ready  |  steps/epoch={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")

In [ ]:
# %% — Cell 9: Training & evaluation helpers
def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
 
    with torch.set_grad_enabled(training):
        for step, batch in enumerate(tqdm(loader, leave=False)):
            iv  = batch["input_values"].to(DEVICE)
            am  = batch["attention_mask"].to(DEVICE)
            lbl = batch["labels"].to(DEVICE)
 
            with autocast(enabled=CFG["MIXED_PREC"]):
                out  = model(iv, attention_mask=am, labels=lbl)
                loss = out["loss"] / CFG["GRAD_ACCUM"]
 
            if training:
                scaler.scale(loss).backward()
                if (step + 1) % CFG["GRAD_ACCUM"] == 0:
                    scaler.unscale_(optimiser)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimiser)
                    scaler.update()
                    scheduler.step()
                    optimiser.zero_grad()
 
            total_loss += loss.item() * CFG["GRAD_ACCUM"]
            preds = out["logits"].argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbl.cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, f1
 
print("✓ Training helpers defined")

In [ ]:
# %% — Cell 10: Main training loop
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
           "train_f1":   [], "val_f1":   []}
best_val_f1   = 0.0
best_ckpt     = os.path.join(CFG["OUTPUT_DIR"], "best_model.pt")
 
print("=" * 60)
print("Training started")
print("=" * 60)
 
for epoch in range(1, CFG["EPOCHS"] + 1):
    t0 = time.time()
 
    # ── Unfreeze backbone at UNFREEZE_EPOCH ──────────────────────────────
    if epoch == CFG["UNFREEZE_EPOCH"]:
        print(f"\n[Epoch {epoch}] ▶ Unfreezing backbone — adding backbone LR={CFG['LR_BASE']}")
        model.unfreeze_backbone()
        optimiser = build_optimiser(unfreeze=True)
        remaining = (CFG["EPOCHS"] - epoch + 1) * steps_per_epoch
        scheduler = get_linear_schedule_with_warmup(optimiser, 0, remaining)
 
    # ── Train ─────────────────────────────────────────────────────────────
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, training=True)
 
    # ── Validate ──────────────────────────────────────────────────────────
    vl_loss, vl_acc, vl_f1 = run_epoch(val_loader, training=False)
 
    # ── Log ───────────────────────────────────────────────────────────────
    elapsed = time.time() - t0
    for k, v in zip(
        ["train_loss","val_loss","train_acc","val_acc","train_f1","val_f1"],
        [tr_loss, vl_loss, tr_acc, vl_acc, tr_f1, vl_f1]
    ):
        history[k].append(v)
 
    flag = " ← best" if vl_f1 > best_val_f1 else ""
    print(
        f"Epoch {epoch:02d}/{CFG['EPOCHS']}  "
        f"| TrLoss {tr_loss:.4f}  TrAcc {tr_acc:.3f}  TrF1 {tr_f1:.3f}"
        f"  |  VlLoss {vl_loss:.4f}  VlAcc {vl_acc:.3f}  VlF1 {vl_f1:.3f}"
        f"  |  {elapsed:.0f}s{flag}"
    )
 
    # ── Save best checkpoint ──────────────────────────────────────────────
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save({
            "epoch":       epoch,
            "state_dict":  model.state_dict(),
            "val_f1":      vl_f1,
            "val_acc":     vl_acc,
            "cfg":         CFG,
        }, best_ckpt)
 
    # ── Memory cleanup ────────────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()
 
print(f"\n✓ Training complete  |  Best Val F1 = {best_val_f1:.4f}")

In [ ]:
# %% — Cell 11: Save training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_x = range(1, CFG["EPOCHS"] + 1)
 
axes[0].plot(epochs_x, history["train_loss"], label="Train")
axes[0].plot(epochs_x, history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend()
 
axes[1].plot(epochs_x, history["train_acc"], label="Train")
axes[1].plot(epochs_x, history["val_acc"],   label="Val")
axes[1].set_title("Accuracy"); axes[1].legend()
 
axes[2].plot(epochs_x, history["train_f1"], label="Train")
axes[2].plot(epochs_x, history["val_f1"],   label="Val")
axes[2].set_title("Weighted F1"); axes[2].legend()
 
plt.tight_layout()
curve_path = os.path.join(CFG["OUTPUT_DIR"], "training_curves.png")
plt.savefig(curve_path, dpi=150)
plt.show()
print(f"✓ Curves saved → {curve_path}")

In [ ]:
# %% — Cell 12: Evaluate on test set (best checkpoint)
print("Loading best checkpoint for test evaluation …")
ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
print(f"   Restored epoch {ckpt['epoch']}  |  Val F1 = {ckpt['val_f1']:.4f}")
 
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        iv  = batch["input_values"].to(DEVICE)
        am  = batch["attention_mask"].to(DEVICE)
        out = model(iv, attention_mask=am)
        all_preds.extend(out["logits"].argmax(-1).cpu().numpy())
        all_labels.extend(batch["labels"].numpy())
 
test_acc = accuracy_score(all_labels, all_preds)
test_f1  = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
print(f"\n── Test Results ──────────────────────────")
print(f"   Accuracy : {test_acc:.4f}")
print(f"   F1 (wt)  : {test_f1:.4f}")
 
label_names = [CFG["LABEL_MAP"][i] for i in range(CFG["NUM_LABELS"])]
print("\n── Per-class Report ──────────────────────")
print(classification_report(all_labels, all_preds, target_names=label_names, zero_division=0))
 
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=label_names,
            yticklabels=label_names, cmap="Blues", ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix — Test  (F1={test_f1:.3f})")
plt.tight_layout()
cm_path = os.path.join(CFG["OUTPUT_DIR"], "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"✓ Confusion matrix saved → {cm_path}")

In [ ]:
# %% — Cell 13: Export final model for inference
export_dir = os.path.join(CFG["OUTPUT_DIR"], "final_model")
os.makedirs(export_dir, exist_ok=True)
 
# Save full model state
torch.save(model.state_dict(), os.path.join(export_dir, "classifier_weights.pt"))
 
# Save feature extractor config
feature_extractor.save_pretrained(export_dir)
 
# Save label map + test metrics
results = {
    "test_accuracy": round(test_acc, 4),
    "test_f1_weighted": round(test_f1, 4),
    "label_map": CFG["LABEL_MAP"],
    "model_id": CFG["MODEL_ID"],
    "best_val_f1": round(best_val_f1, 4),
}
with open(os.path.join(export_dir, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
 
print(f"✓ Final model exported → {export_dir}")
print(json.dumps(results, indent=2))